# Analysis of the JEPA_10k

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import torch

Import of the latents

In [ ]:

list_results = torch.load("C:\\Users\\alexa\\OneDrive\\Bureau\\JEPA_4_PLM\\latents\\JEPA_10k\\cont_targ_lat_10.pt")

print(list_results[0].keys())
print("length of list_results: ", len(list_results))

dict_keys(['prot_name', 'context_latent', 'target_latent', 'full_seq', 'masked_seq'])
lenght of list_results:  10


In [38]:
print(list_results[0]["context_latent"].shape)

# Remove the batch dimension for the whole list of results
for i in range(len(list_results)):
    list_results[i]["context_latent"] = list_results[i]["context_latent"].squeeze(0)
    list_results[i]["target_latent"] = list_results[i]["target_latent"].squeeze(0)

print(list_results[0]["context_latent"].shape)

torch.Size([348, 320])
torch.Size([348, 320])


Analysis ideas:
- Compare the context and target latents using cosine similarity or Euclidean distance.
- Visualize the latent space using dimensionality reduction techniques like PCA or t-SNE.

## Cosine similarity between context and target latents

Check average cosine similarity

In [47]:
from sklearn.metrics.pairwise import cosine_similarity

for i in range(len(list_results)):
    context_latents = list_results[i]["context_latent"].cpu().numpy()
    target_latents = list_results[i]["target_latent"].cpu().numpy()
    cos_sim = cosine_similarity(context_latents, target_latents)
    euclidean_dist = torch.cdist(list_results[i]["context_latent"], list_results[i]["target_latent"], p=2)
    average_cos_sim = cos_sim.mean()
    average_euclidean_dist = euclidean_dist.mean()
    print(f"Average cosine similarity for {list_results[i]['prot_name']}: {average_cos_sim}")
    # print(f"Average Euclidean distance for {list_results[i]['prot_name']}: {average_euclidean_dist}")
    # print(f"Cosine similarity for {list_results[i]['prot_name']}: {cos_sim}")

Average cosine similarity for UniRef50_B9TJK4: 0.9076655507087708
Average cosine similarity for UniRef50_P12264: 0.9186285734176636
Average cosine similarity for UniRef50_A0A1M5DL89: 0.915614664554596
Average cosine similarity for UniRef50_A0A396HAN0: 0.9504802227020264
Average cosine similarity for UniRef50_I2NE94: 0.9433192610740662
Average cosine similarity for UniRef50_A0A817SU47: 0.9183611869812012
Average cosine similarity for UniRef50_A0A1V4J234: 0.9295344948768616
Average cosine similarity for UniRef50_A0A0G3GTD6: 0.9281708598136902
Average cosine similarity for UniRef50_A0A221KA17: 0.9001253247261047
Average cosine similarity for UniRef50_A0A8H7ZNU4: 0.929295003414154


Check variance by dim

In [57]:
# Variance by dimension
for i in range(len(list_results)):
    context_latents = list_results[i]["context_latent"].cpu().numpy()
    target_latents = list_results[i]["target_latent"].cpu().numpy()
    context_variance = context_latents.var(axis=0)
    target_variance = target_latents.var(axis=0)
    dead_dims = (target_variance < 1e-4).sum()
    print("Number of dead dimensions for {}: {}".format(list_results[i]['prot_name'], dead_dims))

Number of dead dimensions for UniRef50_B9TJK4: 0
Number of dead dimensions for UniRef50_P12264: 0
Number of dead dimensions for UniRef50_A0A1M5DL89: 0
Number of dead dimensions for UniRef50_A0A396HAN0: 0
Number of dead dimensions for UniRef50_I2NE94: 0
Number of dead dimensions for UniRef50_A0A817SU47: 0
Number of dead dimensions for UniRef50_A0A1V4J234: 0
Number of dead dimensions for UniRef50_A0A0G3GTD6: 0
Number of dead dimensions for UniRef50_A0A221KA17: 0
Number of dead dimensions for UniRef50_A0A8H7ZNU4: 0


Distribution of cosine similarity intra-prot

In [62]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def extract_and_pool(latent_tensor):
    """
    Convertit le tenseur en array numpy et s'assure qu'il est au format 1D.
    Si vos latents contiennent une dimension 'séquence' (ex: Longueur x Dimension),
    on applique un mean-pooling pour obtenir un seul vecteur par protéine.
    """
    arr = latent_tensor.cpu().numpy()
    if arr.ndim == 2 and arr.shape[0] > 1 and arr.shape[1] > 1:
        arr = arr.mean(axis=0)  # Mean-pooling sur la longueur de la protéine
    return arr.flatten()

# --- 1. Préparation des données (Vectorisation) ---
context_embeddings = []
target_embeddings = []
prot_names = []

for res in list_results:
    context_embeddings.append(extract_and_pool(res["context_latent"]))
    target_embeddings.append(extract_and_pool(res["target_latent"]))
    prot_names.append(res.get("prot_name", "Unknown"))

# Création des matrices globales (N_proteines, D_dimensions)
X_context = np.array(context_embeddings)
X_target = np.array(target_embeddings)

print(f"📊 Données chargées : {X_context.shape[0]} protéines | Dimension latente : {X_context.shape[1]}")
print("-" * 60)


# --- 2. Analyse de l'Effondrement Global (Informationnel) ---
# On calcule la matrice de similarité cosinus de toutes les paires de contextes
cos_sim_matrix = cosine_similarity(X_context)

# On masque la diagonale (car la similarité d'une protéine avec elle-même est de 1)
n_samples = cos_sim_matrix.shape[0]
mask_hors_diag = ~np.eye(n_samples, dtype=bool)
paires_sim = cos_sim_matrix[mask_hors_diag]

avg_sim_inter = paires_sim.mean()
std_sim_inter = paires_sim.std()

print("🔍 1. EFFONDREMENT GLOBAL (INFORMATIONNEL)")
print(f"➡️ Similarité cosinus moyenne inter-protéines : {avg_sim_inter:.4f}")
print(f"➡️ Écart-type de la similarité inter-protéines  : {std_sim_inter:.4f}")

if avg_sim_inter > 0.85:
    print("❌ ALERTE : Effondrement global détecté ! Le modèle projette presque toutes les protéines sur le même vecteur.")
elif std_sim_inter < 0.05:
    print("⚠️ ATTENTION : Faible diversité. Les représentations sont trop proches les unes des autres.")
else:
    print("✅ Statut : Bon. Le modèle discrimine correctement les différentes protéines.")
print("-" * 60)


# --- 3. Analyse de l'Effondrement Structurel (Dimensionnel) ---
# Analyse de la variance par dimension
variances = np.var(X_context, axis=0)
dead_dims = np.sum(variances < 1e-4)

# Analyse de la dimensionnalité effective via SVD (Singular Value Decomposition)
X_centered = X_context - X_context.mean(axis=0)
_, s, _ = np.linalg.svd(X_centered, full_matrices=False)
s_norm = s / s.sum()
cum_variance = np.cumsum(s_norm)
# Trouver combien de dimensions portent 95% de l'information
effective_dim_95 = np.argmax(cum_variance >= 0.95) + 1

print("📐 2. EFFONDREMENT STRUCTUREL (DIMENSIONNEL)")
print(f"➡️ Dimensions 'mortes' (Variance ~ 0) : {dead_dims} / {X_context.shape[1]}")
print(f"➡️ Dimensions effectives requises pour 95% de la variance : {effective_dim_95} / {min(X_context.shape)}")

if effective_dim_95 < 5:
    print("❌ ALERTE : Effondrement dimensionnel sévère ! Le modèle utilise un sous-espace minuscule.")
else:
    print("✅ Statut : Bon. L'espace latent exploite bien sa géométrie sans se contracter sur une seule ligne/plan.")
print("-" * 60)


# --- 4. Analyse de l'Alignement (Contexte vs Target) ---
# Dans un JEPA, le contexte doit être proche de sa propre cible (target)
# On calcule la similarité cosinus croisée
cross_sim = cosine_similarity(X_context, X_target)
# La diagonale représente l'alignement d'une même protéine
intra_sim = np.diag(cross_sim)

print("🔗 3. ALIGNEMENT CONTEXTE / TARGET")
print(f"➡️ Similarité moyenne Contexte ↔ Target (Même protéine) : {intra_sim.mean():.4f}")

if intra_sim.mean() < 0.4:
    print("⚠️ ATTENTION : Le contexte et la target ne s'alignent pas bien. La tâche prédictive n'est pas apprise.")
else:
    print("✅ Statut : Bon. Le bloc contexte sait prédire les propriétés de la target.")
print("-" * 60)

📊 Données chargées : 10 protéines | Dimension latente : 320
------------------------------------------------------------
🔍 1. EFFONDREMENT GLOBAL (INFORMATIONNEL)
➡️ Similarité cosinus moyenne inter-protéines : 0.9960
➡️ Écart-type de la similarité inter-protéines  : 0.0016
❌ ALERTE : Effondrement global détecté ! Le modèle projette presque toutes les protéines sur le même vecteur.
------------------------------------------------------------
📐 2. EFFONDREMENT STRUCTUREL (DIMENSIONNEL)
➡️ Dimensions 'mortes' (Variance ~ 0) : 4 / 320
➡️ Dimensions effectives requises pour 95% de la variance : 8 / 10
✅ Statut : Bon. L'espace latent exploite bien sa géométrie sans se contracter sur une seule ligne/plan.
------------------------------------------------------------
🔗 3. ALIGNEMENT CONTEXTE / TARGET
➡️ Similarité moyenne Contexte ↔ Target (Même protéine) : 0.9933
✅ Statut : Bon. Le bloc contexte sait prédire les propriétés de la target.
--------------------------------------------------------